# Predict on New Data with OctoPredictor

Octopus trains models using nested cross-validation. Test data for each outer split is stored within the study and used by `OctoTestEvaluator` for post-hoc evaluation (see the analysis notebook).

`OctoPredictor` serves a different purpose: scoring **new, external data** that was never part of the study. The data must come from a different source, a different time period, or a genuinely held-out population. If any sample in the prediction data was seen by any model during fit, cross-validation, or hyperparameter tuning, the resulting scores are invalid.

This notebook demonstrates the full deployment workflow.

| | OctoPredictor |
|---|---|
| **Purpose** | Score genuinely new data that was **never** part of training |
| **Data source** | External: new batch, new CSV, new API response |
| **Prediction** | Ensemble average across all outer-split models |
| **Deployment** | Can be saved/loaded standalone |
| **Use case** | Production inference, scoring new samples |

## 1. Prepare Data

Before training, we set aside a portion of data to simulate new samples arriving later. In a real deployment this step does not exist -- you train on all available data and new data arrives naturally. Here we hold out 20% to have something to predict on.

The held-out data is saved to a CSV file and reloaded later as if it came from an external source. The study never sees this data during training.

In [ ]:
from sklearn.model_selection import train_test_split

from octopus.example_data import load_breast_cancer_data

df, features, _ = load_breast_cancer_data()

# Hold out 20% BEFORE any octopus code runs
train_df, external_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df["target"]
)

# Save external data to CSV (simulates a separate data source)
external_csv = "new_patient_batch.csv"
external_df.to_csv(external_csv, index=False)

print(f"Training data: {len(train_df)} samples")
print(f"External data: {len(external_df)} samples (saved to {external_csv})")

## 2. Train a Study

Train a classification study on the training data. In practice this step happens once -- you train the model, validate it (see the analysis notebook), and then deploy it for ongoing scoring.

In [ ]:
from octopus.modules import Octo
from octopus.study import OctoClassification
from octopus.types import ModelName

study = OctoClassification(
    study_name="predict_demo",
    studies_directory="./studies",
    target_metric="ACCBAL",
    feature_cols=features[:10],
    target_col="target",
    sample_id_col="index",
    stratification_col="target",
    workflow=[
        Octo(
            description="baseline",
            task_id=0,
            models=[ModelName.ExtraTreesClassifier],
            n_trials=25,
        ),
    ],
)

study.fit(data=train_df)
print(f"Study trained at: {study.output_path}")

## 3. Load Study and Inspect Workflow

Before creating a predictor, load the study metadata with `load_study_information()`. This returns a `StudyInfo` object used by all downstream functions.

An octopus study can contain multiple workflow tasks (e.g. feature selection followed by model training, or multiple model configurations). **`OctoPredictor` operates at the task level** -- you need to select which task to use for predictions. Use `workflow_graph()` to visualise the task structure and identify the task ID you want.

In [ ]:
from octopus.poststudy import load_study_information
from octopus.poststudy.tables import workflow_graph

study_info = load_study_information(str(study.output_path))
print(workflow_graph(study_info))

## 4. Load New Data

> **The data you pass to OctoPredictor must be genuinely new.** No sample may have been used during
> training, cross-validation, or hyperparameter tuning. If it was, predictions and scores are
> scientifically invalid.

Here we reload the CSV saved in step 1. In your own workflow, replace this with your actual data source (e.g. `new_df = pd.read_csv("data_from_lab_system.csv")`).

In [ ]:
import pandas as pd

# In practice: new_df = pd.read_csv("data_from_lab_system.csv")
new_df = pd.read_csv(external_csv)
print(f"Loaded {len(new_df)} new samples")
new_df.head()

## 5. Create an OctoPredictor

`OctoPredictor` takes a `StudyInfo` and a `task_id`. It loads the fitted models for that task from all outer splits. Since we already loaded `study_info` above, we pass it directly together with the task ID identified from the workflow graph.

In [ ]:
from octopus.poststudy import OctoPredictor

predictor = OctoPredictor(study_info=study_info, task_id=0)

print(f"ML type:        {predictor.study_info.ml_type}")
print(f"Target metric:  {predictor.study_info.target_metric}")
print(f"Outer splits:   {predictor.study_info.n_outersplits}")
print(f"Feature cols:   {len(predictor.feature_cols)}")

## 6. Predict

Each outer-split model predicts independently on the new data, then results are ensemble-averaged. `predict()` returns a numpy array by default, or a DataFrame with per-split detail when `df=True`.

In [ ]:
predictions = predictor.predict(new_df)
print(f"Predictions shape: {predictions.shape}")
print(f"First 10: {predictions[:10]}")

In [ ]:
predictions_df = predictor.predict(new_df, df=True)
predictions_df.head(10)

## 7. Predict Probabilities

For classification tasks, `predict_proba()` returns class probabilities averaged across outer-split models. Each model produces its own probability estimates, and the ensemble average provides more calibrated probabilities than any single model.

In [ ]:
proba = predictor.predict_proba(new_df)
print(f"Probabilities shape: {proba.shape}")
print(f"Class labels: {predictor.classes_}")

In [ ]:
proba_df = predictor.predict_proba(new_df, df=True)
proba_df.head(10)

## 8. Evaluate Performance (Optional)

If the new data includes ground-truth labels (e.g. from a follow-up study, retrospective labelling, or quality control), you can evaluate how well the model performs on the external data.

This is optional -- in many deployment scenarios, labels are not available at prediction time.

In [ ]:
perf = predictor.performance(new_df, metrics=["AUCROC", "ACCBAL", "F1"])
perf

## 9. Feature Importance (Optional)

`calculate_fi()` computes feature importance on the provided data. For permutation FI, the predictor also loads the per-split training data from the study directory to build the replacement pool. In deployment mode (after `save`/`load`), the user-provided data is used as the pool instead.

Here we use `n_repeats=20` for demonstration. This is too low for production use -- increase to 100+ for stable importance estimates and reliable p-values.

In [ ]:
fi = predictor.calculate_fi(new_df, fi_type="permutation", n_repeats=20)
fi[fi["fi_source"] == "ensemble"].sort_values("importance_mean", ascending=False).head(10)

## 10. Save and Load for Deployment

Save the predictor to a standalone directory. This bundles models and metadata -- no study directory needed afterward. Ship this artifact to a different machine, a Docker container, or a scoring service.

In [ ]:
save_dir = "./deployed_predictor"
predictor.save(save_dir)
print(f"Saved to: {save_dir}")

In [ ]:
import numpy as np

loaded = OctoPredictor.load(save_dir)

loaded_preds = loaded.predict(new_df)
np.testing.assert_allclose(predictions, loaded_preds)
print("Predictions match after save/load round-trip.")

## 11. Cleanup

In [ ]:
import shutil
from pathlib import Path

Path(external_csv).unlink(missing_ok=True)
shutil.rmtree(str(study.output_path), ignore_errors=True)
shutil.rmtree(save_dir, ignore_errors=True)